In [1]:
%run "./00_config.ipynb"

JAVA_HOME: C:\Java\jdk-17
java.exe found at: C:\Java\jdk-17\bin\java.EXE
PySpark home: c:\Users\Asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark
bin dir exists: True
['beeline', 'beeline.cmd', 'docker-image-tool.sh', 'find-spark-home', 'find-spark-home.cmd', 'load-spark-env.cmd', 'load-spark-env.sh', 'pyspark', 'pyspark.cmd', 'pyspark2.cmd', 'run-example', 'run-example.cmd', 'spark-class', 'spark-class.cmd', 'spark-class2.cmd', 'spark-connect-shell', 'spark-shell', 'spark-shell.cmd', 'spark-shell2.cmd', 'spark-sql', 'spark-sql.cmd', 'spark-sql2.cmd', 'spark-submit', 'spark-submit.cmd', 'spark-submit2.cmd', 'sparkR', 'sparkR.cmd', 'sparkR2.cmd']
SPARK_HOME env: None
SPARK_HOME: None
JAVA_HOME  : C:\Java\jdk-17
HADOOP_HOME: C:\hadoop
winutils found at: C:\hadoop\bin\winutils.exe
Ready: C:\covid_pipeline\bronze
Ready: C:\covid_pipeline\silver
Ready: C:\covid_pipeline\gold
Spark version: 3.5.3
Spark master : local[*]
Parquet write test succeeded at: C:\covid_pipeline\

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(RAW_OXCGRT)
)

print("Raw row count:", df_raw.count())
print(df_raw.select("CountryCode", "Jurisdiction", "Date").show(3))

Raw row count: 239661
+-----------+------------+--------+
|CountryCode|Jurisdiction|    Date|
+-----------+------------+--------+
|        AUS|   NAT_TOTAL|20200101|
|        AUS|   NAT_TOTAL|20200102|
|        AUS|   NAT_TOTAL|20200103|
+-----------+------------+--------+
only showing top 3 rows

None


In [3]:
df_clean = (
    df_raw
    .filter(F.col("Jurisdiction") == "NAT_TOTAL")
    .withColumn("date", F.to_date(F.col("Date").cast("string"), "yyyyMMdd"))
    .select(
        F.col("CountryCode").alias("iso_code"),
        F.col("CountryName").alias("country"),
        "date",
        F.col("StringencyIndex_Average").alias("stringency_index"),
        F.col("GovernmentResponseIndex_Average").alias("govt_response_index"),
        F.col("ContainmentHealthIndex_Average").alias("containment_health_index"),
        F.col("EconomicSupportIndex").alias("economic_support_index"),
    )
)

print("Cleaned row count (NAT_TOTAL only):", df_clean.count())
df_clean.show(5)

Cleaned row count (NAT_TOTAL only): 7672
+--------+---------+----------+----------------+-------------------+------------------------+----------------------+
|iso_code|  country|      date|stringency_index|govt_response_index|containment_health_index|economic_support_index|
+--------+---------+----------+----------------+-------------------+------------------------+----------------------+
|     AUS|Australia|2020-01-01|             0.0|                0.0|                     0.0|                   0.0|
|     AUS|Australia|2020-01-02|             0.0|                0.0|                     0.0|                   0.0|
|     AUS|Australia|2020-01-03|             0.0|                0.0|                     0.0|                   0.0|
|     AUS|Australia|2020-01-04|             0.0|                0.0|                     0.0|                   0.0|
|     AUS|Australia|2020-01-05|             0.0|                0.0|                     0.0|                   0.0|
+--------+---------+---

In [4]:
# Sanity check: confirm no duplicate iso_code+date after filtering to NAT_TOTAL
dupes = df_clean.groupBy("iso_code", "date").count().filter("count > 1")
print("Duplicate iso_code+date rows remaining:", dupes.count())

Duplicate iso_code+date rows remaining: 0


In [5]:
df_clean.write.mode("overwrite").parquet(SILVER_OXCGRT)
print("Written to:", SILVER_OXCGRT)

Written to: C:\covid_pipeline\silver\oxcgrt
